In [15]:
import pandas as pd 
import numpy as np
import xgboost as xgb
import lightgbm as lgb
import catboost as cb
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.compose import ColumnTransformer
from sklearn.metrics import log_loss

In [5]:
results = pd.read_parquet('../data/interim/results_clean.parquet')
print(results.head())
print(results.shape)

        date home_team away_team  home_score  away_score tournament     city  \
0 1872-11-30  Scotland   England           0           0   Friendly  Glasgow   
1 1873-03-08   England  Scotland           4           2   Friendly   London   
2 1874-03-07  Scotland   England           2           1   Friendly  Glasgow   
3 1875-03-06   England  Scotland           2           2   Friendly   London   
4 1876-03-04  Scotland   England           3           0   Friendly  Glasgow   

    country  neutral  
0  Scotland    False  
1   England    False  
2  Scotland    False  
3   England    False  
4  Scotland    False  
(49482, 9)


In [11]:
# Same data scope + split as Model 1, so both models are evaluated on the identical eval set
model_df = results[results['date'].dt.year >= 1970].reset_index(drop=True)

In [18]:
# As-of Elo feature: run Elo chronologically, record each team's rating BEFORE the match
def add_elo_features(matches, k=30, base_rating=1500):
    # Initialize ratings and elo lists
    ratings = {}
    elo_home_list = []
    elo_away_list = []

    for _, row in matches.iterrows():
        home_team = row['home_team']
        away_team = row['away_team']

        # Get current ratings or initialize them
        elo_home = ratings.get(home_team, base_rating)
        elo_away = ratings.get(away_team, base_rating)

        elo_home_list.append(elo_home)
        elo_away_list.append(elo_away)

        # Calculate expected 
        expected_home = 1 / (1 + 10 ** ((elo_away - elo_home) / 400))
        expected_away = 1 - expected_home

        # Determine actual result
        if row['home_score'] > row['away_score']:
            actual_home = 1
            actual_away = 0
        elif row['home_score'] == row['away_score']:
            actual_home = 0.5
            actual_away = 0.5
        else:
            actual_home = 0
            actual_away = 1

        # Add change
        change = k * (actual_home - expected_home)
        ratings[home_team] = elo_home + change
        ratings[away_team] = elo_away - change

    matches = matches.copy()
    matches['elo_home'] = elo_home_list
    matches['elo_away'] = elo_away_list
    
    return matches

model_df = add_elo_features(model_df)   # model_df must be chronologically sorted (it is)
print(model_df[['date', 'home_team', 'away_team', 'elo_home', 'elo_away']].head())

        date home_team       away_team  elo_home  elo_away
0 1970-01-04     Malta      Luxembourg    1500.0    1500.0
1 1970-01-14   England     Netherlands    1500.0    1500.0
2 1970-01-28    Israel     Netherlands    1500.0    1500.0
3 1970-02-04      Peru  Czechoslovakia    1500.0    1500.0
4 1970-02-06  Cameroon     Ivory Coast    1500.0    1500.0


In [20]:
# Leakage-safe split — boundaries mutually exclusive so no match is in two slices
wc2026_mask = (model_df['tournament'] == 'FIFA World Cup') & (model_df['date'].dt.year == 2026)

train_df = model_df[model_df['date'] < '2024-01-01']                   
eval_df = model_df[(model_df['date'] >= '2024-01-01') & ~wc2026_mask]  # held-out test set (WC excluded)
wc2026_df = model_df[wc2026_mask]                                      # forecast target, held aside

print(f"train {len(train_df)} | eval {len(eval_df)} | wc2026 {len(wc2026_df)}")
print(train_df.columns) # New features passed successfully

train 38900 | eval 2543 | wc2026 79
Index(['date', 'home_team', 'away_team', 'home_score', 'away_score',
       'tournament', 'city', 'country', 'neutral', 'elo_home', 'elo_away'],
      dtype='str')


In [22]:
print(model_df.head())

        date home_team       away_team  home_score  away_score  \
0 1970-01-04     Malta      Luxembourg           1           1   
1 1970-01-14   England     Netherlands           0           0   
2 1970-01-28    Israel     Netherlands           0           1   
3 1970-02-04      Peru  Czechoslovakia           0           2   
4 1970-02-06  Cameroon     Ivory Coast           3           2   

               tournament      city  country  neutral  elo_home  elo_away  
0                Friendly     Gżira    Malta    False    1500.0    1500.0  
1                Friendly    London  England    False    1500.0    1500.0  
2                Friendly     Jaffa   Israel    False    1500.0    1500.0  
3                Friendly      Lima     Peru    False    1500.0    1500.0  
4  African Cup of Nations  Khartoum    Sudan     True    1500.0    1500.0  


In [31]:
# Reshape to a per-team-per-match table for computing rolling form
def reshape_to_long_format(df):
    home_rows = df.rename(columns={
        'home_team': 'team',
        'home_score': 'goals_scored',
        'away_score': 'goals_conceded'
    })[['date', 'team', 'goals_scored', 'goals_conceded']]

    away_rows = df.rename(columns={
        'away_team': 'team',
        'away_score': 'goals_scored',
        'home_score': 'goals_conceded'
    })[['date', 'team', 'goals_scored', 'goals_conceded']]

    long_format = pd.concat([home_rows, away_rows], ignore_index=True)
    return long_format

print(reshape_to_long_format(model_df).head())

        date      team  goals_scored  goals_conceded
0 1970-01-04     Malta             1               1
1 1970-01-14   England             0               0
2 1970-01-28    Israel             0               1
3 1970-02-04      Peru             0               2
4 1970-02-06  Cameroon             3               2
